In [83]:
print("bonjour")

bonjour


In [84]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

RAW = Path("../data/raw")

In [85]:
lignes = []

for f in sorted(RAW.rglob("*")):
    if f.is_file() and not f.name.startswith("~$"):
        lignes.append({
            "chemin": f,
            "fichier": f.name,
            "annee": f.relative_to(RAW).parts[0],
            "type": f.suffix,
        })

inv = pd.DataFrame(lignes)
inv

,chemin,fichier,annee,type
0,..\data\raw\2021\ETU-2021-01 Optimisation du s...,ESS-2021-001.xlsx,2021,.xlsx
1,..\data\raw\2021\ETU-2021-01 Optimisation du s...,ESS-2021-002.xlsx,2021,.xlsx
2,..\data\raw\2021\ETU-2021-01 Optimisation du s...,ESS-2021-003.xlsx,2021,.xlsx
3,..\data\raw\2021\ETU-2021-01 Optimisation du s...,ESS-2021-004 - copie.xlsx,2021,.xlsx
4,..\data\raw\2021\ETU-2021-01 Optimisation du s...,ESS-2021-004.xlsx,2021,.xlsx
...,...,...,...,...
57,..\data\raw\2024\ETU-2024-03 Effet de la charg...,ESS-2024-045.xlsx,2024,.xlsx
58,..\data\raw\2024\ETU-2024-03 Effet de la charg...,ESS-2024-046.xlsx,2024,.xlsx
59,..\data\raw\2024\ETU-2024-03 Effet de la charg...,ESS-2024-047.xlsx,2024,.xlsx
60,..\data\raw\2024\ETU-2024-03 Effet de la charg...,ESS-2024-048.xlsx,2024,.xlsx


In [86]:
print(len(inv), "fichiers")
print(inv["type"].value_counts())
pd.crosstab(inv["annee"], inv["type"])

62 fichiers
type
.xlsx    50
.docx    12
Name: count, dtype: int64


type,.docx,.xlsx
annee,,
2021,3,15
2022,3,10
2023,3,11
2024,3,14


In [87]:
def detecter_gabarit(chemin):
    if chemin.suffix == ".docx":
        return "rapport_word"
    feuilles = load_workbook(chemin, read_only=True).sheetnames
    if "Formulation" in feuilles and "Résultats" in feuilles:
        return "nouveau"
    if "Essai" in feuilles:
        return "ancien"
    return "inconnu"

inv["gabarit"] = inv["chemin"].apply(detecter_gabarit)
inv["gabarit"].value_counts()

gabarit
ancien          25
nouveau         24
rapport_word    12
inconnu          1
Name: count, dtype: int64

In [88]:
inconnus = inv[inv["gabarit"] == "inconnu"]
print("Fichiers non reconnus :")
print(inconnus["fichier"])

copies = inv[inv["fichier"].str.contains("copie")]
print("Doublons possibles :")
print(copies["fichier"])

Fichiers non reconnus :
44    notes_diverses.xlsx
Name: fichier, dtype: str
Doublons possibles :
3    ESS-2021-004 - copie.xlsx
Name: fichier, dtype: str


In [89]:
def trouver_ligne(ws, mot):
    for i in range(1, ws.max_row + 1):
        if ws.cell(i, 1).value == mot:
            return i
    return None

In [90]:
wb = load_workbook(sorted(RAW.rglob("ESS-2024-035.xlsx"))[0])
print(trouver_ligne(wb["Formulation"], "Composant"))   # doit afficher 6
print(trouver_ligne(wb["Résultats"], "Propriété"))      # doit afficher 3
print(trouver_ligne(wb["Formulation"], "Banane"))       # doit afficher None

6
3
None


In [91]:
wb = load_workbook(sorted(RAW.rglob("ESS-2024-035.xlsx"))[0], data_only=True)
ws = wb["Formulation"]

essai = ws["B2"].value
debut = trouver_ligne(ws, "Composant") + 1

formulation = []
for i in range(debut, ws.max_row + 1):
    composant = ws.cell(i, 1).value
    quantite = ws.cell(i, 2).value
    if composant is None:
        continue
    formulation.append({
        "essai": essai,
        "composant": composant,
        "quantite": quantite,
        "source": f"Formulation!B{i}",
    })

pd.DataFrame(formulation)

,essai,composant,quantite,source
0,ESS-2024-035,PVC pâte,100,Formulation!B7
1,ESS-2024-035,DINP,50,Formulation!B8
2,ESS-2024-035,stab Ca/Zn,1,Formulation!B9
3,ESS-2024-035,Carbonate de calcium,20,Formulation!B10


In [92]:
ws = wb["Résultats"]

debut = trouver_ligne(ws, "Propriété") + 1

mesures = []
for i in range(debut, ws.max_row + 1):
    propriete = ws.cell(i, 1).value
    valeur = ws.cell(i, 2).value
    unite = ws.cell(i, 3).value
    if propriete is None:
        continue
    mesures.append({
        "essai": essai,
        "propriete": propriete,
        "valeur": valeur,
        "unite": unite,
        "source": f"Résultats!B{i}",
    })

pd.DataFrame(mesures)

,essai,propriete,valeur,unite,source
0,ESS-2024-035,Viscosité 1h,5833.0,mPa.s,Résultats!B4
1,ESS-2024-035,Viscosité 24h,7152.0,mPa.s,Résultats!B5
2,ESS-2024-035,Dureté Shore A,80.5,Shore A,Résultats!B6
3,ESS-2024-035,Stabilité thermique,22.6,min,Résultats!B7


In [93]:
def lire_nouveau(chemin):
    wb = load_workbook(chemin, data_only=True)

    # page Formulation
    ws = wb["Formulation"]
    essai = ws["B2"].value
    debut = trouver_ligne(ws, "Composant") + 1
    formulation = []
    for i in range(debut, ws.max_row + 1):
        composant = ws.cell(i, 1).value
        if composant is None:
            continue
        formulation.append({
            "essai": essai,
            "composant": composant,
            "quantite": ws.cell(i, 2).value,
            "source": f"{chemin.name} [Formulation!B{i}]",
        })

    # page Résultats
    ws = wb["Résultats"]
    debut = trouver_ligne(ws, "Propriété") + 1
    mesures = []
    for i in range(debut, ws.max_row + 1):
        propriete = ws.cell(i, 1).value
        if propriete is None:
            continue
        mesures.append({
            "essai": essai,
            "propriete": propriete,
            "valeur": ws.cell(i, 2).value,
            "unite": ws.cell(i, 3).value,
            "source": f"{chemin.name} [Résultats!B{i}]",
        })

    return formulation, mesures

In [94]:
fichier = sorted(RAW.rglob("ESS-2024-035.xlsx"))[0]
formulation, mesures = lire_nouveau(fichier)

display(pd.DataFrame(formulation))
display(pd.DataFrame(mesures))

,essai,composant,quantite,source
0,ESS-2024-035,PVC pâte,100,ESS-2024-035.xlsx [Formulation!B7]
1,ESS-2024-035,DINP,50,ESS-2024-035.xlsx [Formulation!B8]
2,ESS-2024-035,stab Ca/Zn,1,ESS-2024-035.xlsx [Formulation!B9]
3,ESS-2024-035,Carbonate de calcium,20,ESS-2024-035.xlsx [Formulation!B10]


,essai,propriete,valeur,unite,source
0,ESS-2024-035,Viscosité 1h,5833.0,mPa.s,ESS-2024-035.xlsx [Résultats!B4]
1,ESS-2024-035,Viscosité 24h,7152.0,mPa.s,ESS-2024-035.xlsx [Résultats!B5]
2,ESS-2024-035,Dureté Shore A,80.5,Shore A,ESS-2024-035.xlsx [Résultats!B6]
3,ESS-2024-035,Stabilité thermique,22.6,min,ESS-2024-035.xlsx [Résultats!B7]


In [95]:
def lire_ancien(chemin):
    wb = load_workbook(chemin, data_only=True)
    ws = wb["Essai"]
    essai = ws["B1"].value

    formulation = []
    mesures = []
    zone = None

    for i in range(1, ws.max_row + 1):
        a = ws.cell(i, 1).value
        b = ws.cell(i, 2).value

        if a is None:
            continue
        if a == "COMPOSITION":
            zone = "formulation"
            continue
        if a == "MESURES":
            zone = "mesures"
            continue
        if a in ("Produit", "Mesure"):
            continue

        source = f"{chemin.name} [Essai!B{i}]"
        if zone == "formulation":
            formulation.append({"essai": essai, "composant": a, "quantite": b, "source": source})
        elif zone == "mesures":
            mesures.append({"essai": essai, "propriete": a, "valeur": b, "unite": None, "source": source})

    return formulation, mesures

In [96]:
chemin = sorted(RAW.rglob("ESS-2022-022.xlsx"))[0]
formulation, mesures = lire_ancien(chemin)

display(pd.DataFrame(formulation))
display(pd.DataFrame(mesures))

,essai,composant,quantite,source
0,ESS-2022-022,Résine PVC pâte,100,ESS-2022-022.xlsx [Essai!B7]
1,ESS-2022-022,DINCH,50,ESS-2022-022.xlsx [Essai!B8]
2,ESS-2022-022,stab Ca/Zn,1,ESS-2022-022.xlsx [Essai!B9]
3,ESS-2022-022,CaCO3,10,ESS-2022-022.xlsx [Essai!B10]


,essai,propriete,valeur,unite,source
0,ESS-2022-022,Visco 1h (Pa.s),5.99,None,ESS-2022-022.xlsx [Essai!B14]
1,ESS-2022-022,Visco 24h (Pa.s),7.43,None,ESS-2022-022.xlsx [Essai!B15]
2,ESS-2022-022,Shore A,78.20,None,ESS-2022-022.xlsx [Essai!B16]
3,ESS-2022-022,Temps de stabilité (s),1386.00,None,ESS-2022-022.xlsx [Essai!B17]


In [97]:
toute_formulation = []
toutes_mesures = []

for i, ligne in inv.iterrows():
    chemin = ligne["chemin"]
    gabarit = ligne["gabarit"]

    if gabarit == "nouveau":
        f, m = lire_nouveau(chemin)
    elif gabarit == "ancien":
        f, m = lire_ancien(chemin)
    else:
        continue

    toute_formulation.extend(f)
    toutes_mesures.extend(m)

df_formulation = pd.DataFrame(toute_formulation)
df_mesures = pd.DataFrame(toutes_mesures)

print(len(df_formulation), "lignes de formulation")
print(len(df_mesures), "mesures")
print(df_formulation["essai"].nunique(), "essais différents")

182 lignes de formulation
196 mesures
48 essais différents


In [98]:
df_mesures["propriete"].value_counts()

propriete
Visco 1h (Pa.s)           25
Visco 24h (Pa.s)          25
Shore A                   25
Temps de stabilité (s)    25
Viscosité 1h              24
Viscosité 24h             24
Dureté Shore A            24
Stabilité thermique       24
Name: count, dtype: int64

In [99]:
df_formulation["composant"].value_counts()


composant
stab Ca/Zn              26
resine PVC pate         24
Stabilisant Ca/Zn       23
Carbonate de calcium    21
Résine PVC pâte         15
CaCO3                   14
PVC pâte                10
DINP                     9
Dinp                     8
Dinch                    8
dotp                     5
plastifiant DOTP         4
DINCH                    4
DINP                     4
DOTP                     3
plastifiant DINP         3
plastifant DINP          1
Name: count, dtype: int64

In [100]:
%pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [101]:
from docx import Document

chemin = sorted(RAW.rglob("ETU-2021-01_rapport.docx"))[0]
doc = Document(chemin)

for p in doc.paragraphs:
    print(p.text)

Étude ETU-2021-01
Document fictif généré pour démonstration.
Titre : Optimisation du stabilisant
Objectif : Évaluer l'effet du stabilisant Ca/Zn sur la stabilité thermique.
Auteur : L. Bernard
Date : 05/06/2021
Conditions : gélification à 170 °C pendant 2 min.
Liste des essais :


In [102]:
texte = "Auteur : L. Bernard"
cle, valeur = texte.split(" : ", 1)
print(cle)
print(valeur)

Auteur
L. Bernard


In [103]:
etude = {"fichier": chemin.name}

for p in doc.paragraphs:
    texte = p.text
    if texte.startswith("Étude "):
        etude["code"] = texte.replace("Étude ", "")
    elif " : " in texte:
        cle, valeur = texte.split(":", 1)
        etude[cle] = valeur

etude

{'fichier': 'ETU-2021-01_rapport.docx',
 'code': 'ETU-2021-01',
 'Titre ': ' Optimisation du stabilisant',
 'Objectif ': " Évaluer l'effet du stabilisant Ca/Zn sur la stabilité thermique.",
 'Auteur ': ' L. Bernard',
 'Date ': ' 05/06/2021',
 'Conditions ': ' gélification à 170 °C pendant 2 min.'}

In [104]:
import re

texte = "gélification à 170 °C pendant 2 min."

temp = re.search(r"(\d+) °C", texte)
duree = re.search(r"(\d+) min", texte)

print(temp.group(1))
print(duree.group(1))

170
2


In [105]:
tableau = doc.tables[0]

for ligne in tableau.rows[1:]:
    print(ligne.cells[0].text)

ESS-2021-001
ESS-2021-002
ESS-2021-003
ESS-2021-004
ESS-2021-005


In [106]:
def lire_word(chemin):
    doc = Document(chemin)

    # 1. les infos de l'étude
    etude = {"fichier": chemin.name}
    for p in doc.paragraphs:
        texte = p.text
        if texte.startswith("Étude "):
            etude["code"] = texte.replace("Étude ", "")
        elif " : " in texte:
            cle, valeur = texte.split(" : ", 1)
            etude[cle] = valeur

    # 2. température et durée
    conditions = etude.get("Conditions", "")
    temp = re.search(r"(\d+) °C", conditions)
    duree = re.search(r"(\d+) min", conditions)
    etude["temperature_C"] = int(temp.group(1)) if temp else None
    etude["duree_min"] = int(duree.group(1)) if duree else None

    # 3. la liste des essais
    essais = []
    for ligne in doc.tables[0].rows[1:]:
        essais.append({"etude": etude["code"], "essai": ligne.cells[0].text})

    return etude, essais

In [107]:
toutes_etudes = []
tous_liens = []

for i, ligne in inv.iterrows():
    if ligne["gabarit"] == "rapport_word":
        e, l = lire_word(ligne["chemin"])
        toutes_etudes.append(e)
        tous_liens.extend(l)

df_etudes = pd.DataFrame(toutes_etudes)
df_liens = pd.DataFrame(tous_liens)

print(len(df_etudes), "études")
print(len(df_liens), "liens étude → essai")
df_etudes[["code", "Titre", "Auteur", "temperature_C", "duree_min"]]

12 études
48 liens étude → essai


,code,Titre,Auteur,temperature_C,duree_min
0,ETU-2021-01,Optimisation du stabilisant,L. Bernard,170,2
1,ETU-2021-02,Effet de la charge minérale,S. Petit,180,3
2,ETU-2021-03,Influence du taux de plastifiant,L. Bernard,190,4
3,ETU-2022-01,Effet de la charge minérale,A. Martin,190,4
4,ETU-2022-02,Influence du taux de plastifiant,C. Dubois,180,3
5,ETU-2022-03,Comparaison de plastifiants,A. Martin,1800,3
6,ETU-2023-01,Influence du taux de plastifiant,A. Martin,170,3
7,ETU-2023-02,Comparaison de plastifiants,C. Dubois,170,4
8,ETU-2023-03,Optimisation du stabilisant,L. Bernard,180,2
9,ETU-2024-01,Comparaison de plastifiants,S. Petit,180,4


In [108]:
df_formulation["composant"].value_counts()

composant
stab Ca/Zn              26
resine PVC pate         24
Stabilisant Ca/Zn       23
Carbonate de calcium    21
Résine PVC pâte         15
CaCO3                   14
PVC pâte                10
DINP                     9
Dinp                     8
Dinch                    8
dotp                     5
plastifiant DOTP         4
DINCH                    4
DINP                     4
DOTP                     3
plastifiant DINP         3
plastifant DINP          1
Name: count, dtype: int64

In [109]:
nomenclature = {
    "pvc pâte": "PVC",
    "résine pvc pâte": "PVC",
    "resine pvc pate": "PVC",
    "dinp": "DINP",
    "plastifiant dinp": "DINP",
    "dotp": "DOTP",
    "plastifiant dotp": "DOTP",
    "dinch": "DINCH",
    "stabilisant ca/zn": "Stabilisant Ca/Zn",
    "stab ca/zn": "Stabilisant Ca/Zn",
    "carbonate de calcium": "CaCO3",
    "caco3": "CaCO3",
}

In [110]:
def normaliser_composant(nom):
    cle = nom.strip().lower()
    return nomenclature.get(cle, None)

print(normaliser_composant("Dinp"))
print(normaliser_composant("  Résine PVC pâte "))
print(normaliser_composant("plastifant DINP"))

DINP
PVC
None


In [111]:
df_formulation["composant_norm"] = df_formulation["composant"].apply(normaliser_composant)

df_formulation["composant_norm"].value_counts(dropna=False)

composant_norm
PVC                  49
Stabilisant Ca/Zn    49
CaCO3                35
DINP                 24
DOTP                 12
DINCH                12
NaN                   1
Name: count, dtype: int64

In [114]:
df_formulation[df_formulation["composant_norm"].isna()]

,essai,composant,quantite,source,composant_norm
102,ESS-2023-027,plastifant DINP,40,ESS-2023-027.xlsx [Formulation!B8],NaN


In [115]:
def separer_unite(ligne):
    texte = ligne["propriete"]
    trouve = re.search(r"\((.+)\)", texte)
    if trouve:
        nom = re.sub(r"\s*\(.+\)", "", texte)
        unite = trouve.group(1)
    else:
        nom = texte
        unite = ligne["unite"]
    return pd.Series([nom, unite])

In [116]:
df_mesures[["nom", "unite_origine"]] = df_mesures.apply(separer_unite, axis=1)
df_mesures[["propriete", "unite", "nom", "unite_origine"]].drop_duplicates("propriete")

,propriete,unite,nom,unite_origine
0,Visco 1h (Pa.s),NaN,Visco 1h,Pa.s
1,Visco 24h (Pa.s),NaN,Visco 24h,Pa.s
2,Shore A,NaN,Shore A,NaN
3,Temps de stabilité (s),NaN,Temps de stabilité,s
100,Viscosité 1h,mPa.s,Viscosité 1h,mPa.s
101,Viscosité 24h,mPa.s,Viscosité 24h,mPa.s
102,Dureté Shore A,Shore A,Dureté Shore A,Shore A
103,Stabilité thermique,min,Stabilité thermique,min


In [117]:
proprietes = {
    "visco 1h": "viscosite_1h",
    "viscosité 1h": "viscosite_1h",
    "visco 24h": "viscosite_24h",
    "viscosité 24h": "viscosite_24h",
    "shore a": "durete_shore_a",
    "dureté shore a": "durete_shore_a",
    "temps de stabilité": "stabilite_thermique",
    "stabilité thermique": "stabilite_thermique",
}

df_mesures["propriete_norm"] = df_mesures["nom"].str.strip().str.lower().map(proprietes)

df_mesures["propriete_norm"].value_counts(dropna=False)

propriete_norm
viscosite_1h           49
viscosite_24h          49
durete_shore_a         49
stabilite_thermique    49
Name: count, dtype: int64

In [118]:
conversions = {
    "Pa.s":    ("mPa.s", 1000),
    "mPa.s":   ("mPa.s", 1),
    "s":       ("min", 1/60),
    "min":     ("min", 1),
    "Shore A": ("Shore A", 1),
}

print(conversions["Pa.s"])

('mPa.s', 1000)


In [119]:
unite_cible, facteur = conversions["Pa.s"]
print(unite_cible)
print(facteur)

mPa.s
1000


In [122]:
def convertir(ligne):
    unite = ligne["unite_origine"]
    if ligne["propriete_norm"] == "durete_shore_a":
        unite = "Shore A"
    if pd.isna(ligne["valeur"]) or unite not in conversions:
        return pd.Series([None, unite])
    unite_cible, facteur = conversions[unite]
    valeur = round(float(ligne["valeur"]) * facteur, 2)
    return pd.Series([valeur, unite_cible])

In [123]:
df_mesures[["valeur_norm", "unite_norm"]] = df_mesures.apply(convertir, axis=1)

df_mesures.groupby(["propriete_norm", "unite_norm"]).size()

propriete_norm       unite_norm
durete_shore_a       Shore A       49
stabilite_thermique  min           49
viscosite_1h         mPa.s         49
viscosite_24h        mPa.s         49
dtype: int64

In [ ]:
df_formulation["quantite_phr"] = pd.to_numeric(
    df_formulation["quantite"].astype(str).str.replace(",", "."),
    errors="coerce"
)

df_formulation[["composant", "quantite", "quantite_phr"]].head(10) 

,composant,quantite,quantite_phr
0,PVC pâte,100,100
1,dotp,40,40
2,Stabilisant Ca/Zn,1,1
3,resine PVC pate,100,100
4,dotp,40,40
5,Stabilisant Ca/Zn,1,1
6,Carbonate de calcium,10,10
7,resine PVC pate,100,100
8,DOTP,60,60
9,stab Ca/Zn,4,4


In [125]:
f_ok = df_formulation[~df_formulation["source"].str.contains("copie")]
m_ok = df_mesures[~df_mesures["source"].str.contains("copie")]

print(len(df_formulation), "→", len(f_ok))
print(len(df_mesures), "→", len(m_ok))

182 → 178
196 → 192


In [126]:
import sqlite3

con = sqlite3.connect("../data/rd_database.db")

etudes = df_etudes.rename(columns={
    "code": "code_etude",
    "Titre": "titre",
    "Objectif": "objectif",
    "Auteur": "auteur",
    "Date": "date",
})

etudes = etudes[["code_etude", "titre", "objectif", "auteur", "date",
                 "temperature_C", "duree_min", "fichier"]]

etudes.to_sql("etude", con, if_exists="replace", index=False)

pd.read_sql("SELECT * FROM etude", con)

,code_etude,titre,objectif,auteur,date,temperature_C,duree_min,fichier
0,ETU-2021-01,Optimisation du stabilisant,Évaluer l'effet du stabilisant Ca/Zn sur la st...,L. Bernard,05/06/2021,170,2,ETU-2021-01_rapport.docx
1,ETU-2021-02,Effet de la charge minérale,Étudier l'ajout de carbonate de calcium sur la...,S. Petit,14/08/2021,180,3,ETU-2021-02_rapport.docx
2,ETU-2021-03,Influence du taux de plastifiant,Mesurer l'effet du taux de plastifiant sur la ...,L. Bernard,13/07/2021,190,4,ETU-2021-03_rapport.docx
3,ETU-2022-01,Effet de la charge minérale,Étudier l'ajout de carbonate de calcium sur la...,A. Martin,06/06/2022,190,4,ETU-2022-01_rapport.docx
4,ETU-2022-02,Influence du taux de plastifiant,Mesurer l'effet du taux de plastifiant sur la ...,C. Dubois,13/02/2022,180,3,ETU-2022-02_rapport.docx
5,ETU-2022-03,Comparaison de plastifiants,Comparer plusieurs plastifiants à taux constant.,A. Martin,08/09/2022,1800,3,ETU-2022-03_rapport.docx
6,ETU-2023-01,Influence du taux de plastifiant,Mesurer l'effet du taux de plastifiant sur la ...,A. Martin,14/04/2023,170,3,ETU-2023-01_rapport.docx
7,ETU-2023-02,Comparaison de plastifiants,Comparer plusieurs plastifiants à taux constant.,C. Dubois,02/05/2023,170,4,ETU-2023-02_rapport.docx
8,ETU-2023-03,Optimisation du stabilisant,Évaluer l'effet du stabilisant Ca/Zn sur la st...,L. Bernard,08/08/2023,180,2,ETU-2023-03_rapport.docx
9,ETU-2024-01,Comparaison de plastifiants,Comparer plusieurs plastifiants à taux constant.,S. Petit,10/01/2024,180,4,ETU-2024-01_rapport.docx


In [ ]:
essais = df_liens.rename(columns={"etude": "code_etude", "essai": "code_essai"})

essais.to_sql("essai", con, if_exists="replace", index=False) 

pd.read_sql("SELECT * FROM essai", con)  

,code_etude,code_essai
0,ETU-2021-01,ESS-2021-001
1,ETU-2021-01,ESS-2021-002
2,ETU-2021-01,ESS-2021-003
3,ETU-2021-01,ESS-2021-004
4,ETU-2021-01,ESS-2021-005
5,ETU-2021-02,ESS-2021-006
6,ETU-2021-02,ESS-2021-007
7,ETU-2021-02,ESS-2021-008
8,ETU-2021-02,ESS-2021-009
9,ETU-2021-02,ESS-2021-010


In [128]:
formulation = f_ok[["essai", "composant", "composant_norm", "quantite_phr", "source"]]
formulation = formulation.rename(columns={"essai": "code_essai"})
formulation.to_sql("formulation", con, if_exists="replace", index=False)

mesure = m_ok[["essai", "propriete", "valeur", "unite_origine",
               "propriete_norm", "valeur_norm", "unite_norm", "source"]]
mesure = mesure.rename(columns={"essai": "code_essai",
                                "propriete": "propriete_origine",
                                "valeur": "valeur_origine"})
mesure.to_sql("mesure", con, if_exists="replace", index=False)

for t in ["etude", "essai", "formulation", "mesure"]:
    print(t, pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", con)["n"][0])

etude 12
essai 48
formulation 178
mesure 192


In [132]:
requete = """
SELECT e.code_essai,
       et.titre,
       f.quantite_phr AS dinp_phr,
       m.valeur_norm  AS visco_1h
FROM essai e
JOIN etude et      ON et.code_etude = e.code_etude
JOIN formulation f ON f.code_essai  = e.code_essai AND f.composant_norm = 'DINP'
JOIN mesure m      ON m.code_essai  = e.code_essai AND m.propriete_norm = 'viscosite_1h'
ORDER BY m.valeur_norm
"""

pd.read_sql(requete, con)

,code_essai,titre,dinp_phr,visco_1h
0,ESS-2023-026,Influence du taux de plastifiant,50,4.6
1,ESS-2024-038,Comparaison de plastifiants,80,2947.0
2,ESS-2021-012,Influence du taux de plastifiant,80,3270.0
3,ESS-2023-030,Comparaison de plastifiants,80,3691.0
4,ESS-2024-037,Comparaison de plastifiants,80,3842.0
5,ESS-2024-042,Optimisation du stabilisant,70,4084.0
6,ESS-2021-014,Influence du taux de plastifiant,80,4120.0
7,ESS-2024-039,Optimisation du stabilisant,80,4147.0
8,ESS-2021-009,Effet de la charge minérale,80,4270.0
9,ESS-2021-011,Influence du taux de plastifiant,70,4610.0


In [133]:
anomalies = []

# 1. Fichiers non reconnus
for _, r in inv[inv["gabarit"] == "inconnu"].iterrows():
    anomalies.append({"categorie": "Fichier non reconnu",
                      "detail": "aucun gabarit connu",
                      "source": r["fichier"]})

# 2. Doublons
for _, r in inv[inv["fichier"].str.contains("copie")].iterrows():
    anomalies.append({"categorie": "Doublon",
                      "detail": "copie d'un essai déjà chargé",
                      "source": r["fichier"]})

# 3. Composants inconnus
for _, r in f_ok[f_ok["composant_norm"].isna()].iterrows():
    anomalies.append({"categorie": "Composant inconnu",
                      "detail": f"'{r['composant']}' absent de la nomenclature",
                      "source": r["source"]})

pd.DataFrame(anomalies)

,categorie,detail,source
0,Fichier non reconnu,aucun gabarit connu,notes_diverses.xlsx
1,Doublon,copie d'un essai déjà chargé,ESS-2021-004 - copie.xlsx
2,Composant inconnu,'plastifant DINP' absent de la nomenclature,ESS-2023-027.xlsx [Formulation!B8]


In [134]:
# 4. Valeurs manquantes
for _, r in m_ok[m_ok["valeur"].isna()].iterrows():
    anomalies.append({"categorie": "Valeur manquante",
                      "detail": f"{r['propriete_norm']} vide pour {r['essai']}",
                      "source": r["source"]})

# 5. Valeurs hors plage
plages = {
    "viscosite_1h":        (500, 20000),
    "viscosite_24h":       (500, 30000),
    "durete_shore_a":      (40, 100),
    "stabilite_thermique": (5, 120),
}

for _, r in m_ok.dropna(subset=["valeur_norm"]).iterrows():
    mini, maxi = plages[r["propriete_norm"]]
    if not (mini <= r["valeur_norm"] <= maxi):
        anomalies.append({"categorie": "Valeur hors plage",
                          "detail": f"{r['propriete_norm']} = {r['valeur_norm']} {r['unite_norm']} (attendu {mini}-{maxi})",
                          "source": r["source"]})

# 6. Température de gélification impossible
for _, r in df_etudes.iterrows():
    if not (140 <= r["temperature_C"] <= 220):
        anomalies.append({"categorie": "Valeur hors plage",
                          "detail": f"gélification à {r['temperature_C']} °C (attendu 140-220)",
                          "source": r["fichier"]})

df_anomalies = pd.DataFrame(anomalies)
df_anomalies

,categorie,detail,source
0,Fichier non reconnu,aucun gabarit connu,notes_diverses.xlsx
1,Doublon,copie d'un essai déjà chargé,ESS-2021-004 - copie.xlsx
2,Composant inconnu,'plastifant DINP' absent de la nomenclature,ESS-2023-027.xlsx [Formulation!B8]
3,Valeur manquante,viscosite_24h vide pour ESS-2023-025,ESS-2023-025.xlsx [Résultats!B5]
4,Valeur hors plage,viscosite_1h = 4.6 mPa.s (attendu 500-20000),ESS-2023-026.xlsx [Résultats!B4]
5,Valeur hors plage,gélification à 1800 °C (attendu 140-220),ETU-2022-03_rapport.docx


In [135]:
df_anomalies.to_sql("anomalie", con, if_exists="replace", index=False)

print(pd.read_sql("SELECT categorie, COUNT(*) AS nombre FROM anomalie GROUP BY categorie", con))

con.close()

             categorie  nombre
0    Composant inconnu       1
1              Doublon       1
2  Fichier non reconnu       1
3    Valeur hors plage       2
4     Valeur manquante       1
